# Fase 3 (CRISP-DM): Preparación de los Datos — Limpieza y Feature Engineering

**Metodología:** CRISP-DM — Fase 3 (*Data Preparation*)  
**Entradas:** `data/01_raw/train-00000-of-00001.parquet` y `data/01_raw/test-00000-of-00001.parquet`  
**Salidas:** `data/02_interim/train_fe.parquet`, `data/02_interim/test_fe.parquet`, `data/03_processed/train_processed.parquet`, `data/03_processed/test_processed.parquet` y `models/imputer_artifacts.joblib`

**Paso 1: Configurar el entorno**

In [ ]:
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
_DEFAULT_MODELS_DIR = ROOT_DIR / 'models'
_DEFAULT_ARTIFACTS_PATH = _DEFAULT_MODELS_DIR / 'imputer_artifacts.joblib'

_NULL_STRINGS = ["nan", "NaN", "None", "null", "<NA>", ""]
_TEXT_COLUMNS = [
    "ID",
    "Delivery_person_ID",
    "Order_Date",
    "Time_Orderd",
    "Time_Order_picked",
    "Weather_conditions",
    "Road_traffic_density",
    "Type_of_order",
    "Type_of_vehicle",
    "Festival",
    "City",
]


def _resolve_artifacts_path(artifacts_path=None):
    if artifacts_path is None:
        return _DEFAULT_ARTIFACTS_PATH
    return Path(artifacts_path)


def clean_data(
    df,
    artifacts=None,
    fit=False,
    artifacts_path=None,
    return_artifacts=False,
):
    """
    Pipeline principal de limpieza de datos sin fuga de datos (Data Leakage).

    Cuando `fit=True` calcula los estadísticos de imputación y centroides de
    ciudad exclusivamente sobre `df` (train) y los persiste en disco. Cuando
    `fit=False` reutiliza `artifacts` o los carga desde `artifacts_path`.
    """
    df = df.copy()
    df = standardize_categoricals(df)
    df = handle_outliers(df)
    df = fix_time_formats(df)

    resolved_path = _resolve_artifacts_path(artifacts_path)
    if fit or (artifacts is None and not resolved_path.exists()):
        artifacts = fit_imputer(df, artifacts_path=resolved_path)
    elif artifacts is None:
        artifacts = load_imputer_artifacts(resolved_path)

    df = fix_coordinates(df, artifacts)
    df = handle_missing_values(df, artifacts)

    if return_artifacts:
        return df, artifacts
    return df


# Paso A1: Estandarización temprana de strings y Ciudad_codigo
def standardize_categoricals(df):
    """Limpia espacios, normaliza nulos de texto y extrae Ciudad_codigo."""
    df = df.copy()
    for col in _TEXT_COLUMNS:
        if col in df.columns:
            cleaned = df[col].astype(str).str.strip()
            df[col] = cleaned.replace(_NULL_STRINGS, np.nan)

    if "Delivery_person_ID" in df.columns:
        extracted = (
            df["Delivery_person_ID"]
            .astype(str)
            .str.extract(r"^([A-Z]+)RES", expand=False)
        )
        if "Ciudad_codigo" in df.columns:
            df["Ciudad_codigo"] = df["Ciudad_codigo"].fillna(extracted).fillna("UNK")
        else:
            df["Ciudad_codigo"] = extracted.fillna("UNK")
    elif "Ciudad_codigo" not in df.columns:
        df["Ciudad_codigo"] = "UNK"

    if "City" in df.columns:
        df["City"] = df["City"].replace("Metropolitian", "Metropolitan")

    return df


# Paso A2: Tratamiento de valores atípicos (Outliers)
def handle_outliers(df):
    """Convierte edades ilegales (<18) en NaN y trunca ratings a [1.0, 5.0]."""
    df = df.copy()
    if "Delivery_person_Age" in df.columns:
        df["Delivery_person_Age"] = pd.to_numeric(
            df["Delivery_person_Age"], errors="coerce"
        )
        df.loc[df["Delivery_person_Age"] < 18, "Delivery_person_Age"] = np.nan

    if "Delivery_person_Ratings" in df.columns:
        df["Delivery_person_Ratings"] = pd.to_numeric(
            df["Delivery_person_Ratings"], errors="coerce"
        )
        df["Delivery_person_Ratings"] = df["Delivery_person_Ratings"].clip(
            lower=1.0, upper=5.0
        )

    if "multiple_deliveries" in df.columns:
        df["multiple_deliveries"] = pd.to_numeric(
            df["multiple_deliveries"], errors="coerce"
        )

    return df


# Paso A3: Reparar formatos de tiempo
def parse_time_to_minutes(val):
    """Convierte HH:MM, HH:MM:SS, 24:XX:00 y decimales de Excel a minutos."""
    if pd.isna(val):
        return np.nan

    if isinstance(val, (int, float, np.number)):
        return float(int(round(float(val) * 1440)) % 1440)

    text = str(val).strip()
    if text in _NULL_STRINGS:
        return np.nan

    if ":" in text:
        parts = text.split(":")
        hours = int(parts[0]) % 24
        minutes = int(parts[1])
        return float(hours * 60 + minutes)

    try:
        return float(int(round(float(text) * 1440)) % 1440)
    except ValueError:
        return np.nan


def fix_time_formats(df):
    """Crea columnas numéricas en minutos desde medianoche para pedido y recogida."""
    df = df.copy()
    if "Time_Orderd" in df.columns:
        df["Time_Orderd_min"] = df["Time_Orderd"].apply(parse_time_to_minutes)
    elif "Time_Orderd_min" not in df.columns:
        df["Time_Orderd_min"] = np.nan

    if "Time_Order_picked" in df.columns:
        df["Time_Order_picked_min"] = df["Time_Order_picked"].apply(
            parse_time_to_minutes
        )
    elif "Time_Order_picked_min" not in df.columns:
        df["Time_Order_picked_min"] = np.nan

    return df


# Paso A4: Ajuste y persistencia de imputadores en train
def fit_imputer(df_train, artifacts_path=None):
    """
    Aprende centroides geográficos, medianas por ciudad y modas exclusivamente
    sobre el conjunto de entrenamiento y los serializa en disco.
    """
    df = df_train.copy()
    if "Ciudad_codigo" not in df.columns:
        df = standardize_categoricals(df)
    if "Delivery_person_Age" in df.columns and (df["Delivery_person_Age"] < 18).any():
        df = handle_outliers(df)
    if "Time_Orderd_min" not in df.columns or "Time_Order_picked_min" not in df.columns:
        df = fix_time_formats(df)

    valid_coords_mask = (
        df["Restaurant_latitude"].abs() >= 1.0
    ) & (df["Restaurant_longitude"].abs() >= 1.0)
    valid_df = df.loc[valid_coords_mask]

    city_centroids = {}
    for city, group in valid_df.groupby("Ciudad_codigo"):
        city_centroids[str(city)] = (
            float(group["Restaurant_latitude"].abs().median()),
            float(group["Restaurant_longitude"].abs().median()),
        )
    global_centroid = (
        float(valid_df["Restaurant_latitude"].abs().median()),
        float(valid_df["Restaurant_longitude"].abs().median()),
    )

    age_by_city = {
        str(k): float(v)
        for k, v in df.groupby("Ciudad_codigo")["Delivery_person_Age"]
        .median()
        .dropna()
        .items()
    }
    ratings_by_city = {
        str(k): float(v)
        for k, v in df.groupby("Ciudad_codigo")["Delivery_person_Ratings"]
        .median()
        .dropna()
        .items()
    }

    valid_times = df.dropna(subset=["Time_Orderd_min", "Time_Order_picked_min"])
    prep_diff = (
        valid_times["Time_Order_picked_min"] - valid_times["Time_Orderd_min"]
    ) % 1440
    prep_time_median = float(prep_diff.median()) if len(prep_diff) > 0 else 10.0

    artifacts = {
        "city_centroids": city_centroids,
        "global_centroid": global_centroid,
        "age_median_by_city": age_by_city,
        "age_median_global": float(df["Delivery_person_Age"].median()),
        "ratings_median_by_city": ratings_by_city,
        "ratings_median_global": float(df["Delivery_person_Ratings"].median()),
        "multiple_deliveries_mode": float(df["multiple_deliveries"].mode().iloc[0]),
        "weather_mode": str(df["Weather_conditions"].mode().iloc[0]),
        "traffic_mode": str(df["Road_traffic_density"].mode().iloc[0]),
        "city_mode": str(df["City"].mode().iloc[0]),
        "prep_time_median": prep_time_median,
    }

    save_imputer_artifacts(artifacts, artifacts_path=artifacts_path)
    return artifacts


def save_imputer_artifacts(artifacts, artifacts_path=None):
    """Guarda los estadísticos de imputación en .joblib y .pkl con pickle."""
    resolved_path = _resolve_artifacts_path(artifacts_path)
    resolved_path.parent.mkdir(parents=True, exist_ok=True)
    with open(resolved_path, "wb") as f:
        pickle.dump(artifacts, f)

    pkl_path = resolved_path.with_suffix(".pkl")
    if pkl_path != resolved_path:
        with open(pkl_path, "wb") as f:
            pickle.dump(artifacts, f)


def load_imputer_artifacts(artifacts_path=None):
    """Carga los estadísticos de imputación previamente guardados."""
    resolved_path = _resolve_artifacts_path(artifacts_path)
    with open(resolved_path, "rb") as f:
        return pickle.load(f)


# Paso A5: Corrección geográfica sin descartar filas
def fix_coordinates(df, artifacts):
    """
    Corrige signos invertidos mediante abs() y traslada los pares (0,0) al
    centroide de su Ciudad_codigo preservando intacto el vector (dlat, dlon).
    """
    df = df.copy()
    coord_cols = [
        "Restaurant_latitude",
        "Restaurant_longitude",
        "Delivery_location_latitude",
        "Delivery_location_longitude",
    ]
    for col in coord_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").abs()

    mask_zero = (df["Restaurant_latitude"] < 1.0) | (df["Restaurant_longitude"] < 1.0)
    if mask_zero.any():
        centroids = artifacts["city_centroids"]
        global_lat, global_lon = artifacts["global_centroid"]

        city_lats = df.loc[mask_zero, "Ciudad_codigo"].map(
            lambda c: centroids.get(str(c), (global_lat, global_lon))[0]
        )
        city_lons = df.loc[mask_zero, "Ciudad_codigo"].map(
            lambda c: centroids.get(str(c), (global_lat, global_lon))[1]
        )

        df.loc[mask_zero, "Restaurant_latitude"] = (
            df.loc[mask_zero, "Restaurant_latitude"] + city_lats.values
        )
        df.loc[mask_zero, "Delivery_location_latitude"] = (
            df.loc[mask_zero, "Delivery_location_latitude"] + city_lats.values
        )
        df.loc[mask_zero, "Restaurant_longitude"] = (
            df.loc[mask_zero, "Restaurant_longitude"] + city_lons.values
        )
        df.loc[mask_zero, "Delivery_location_longitude"] = (
            df.loc[mask_zero, "Delivery_location_longitude"] + city_lons.values
        )

    return df


# Paso A6: Imputación de valores faltantes usando artifacts
def handle_missing_values(df, artifacts):
    """Imputa nulos numéricos, categóricos y temporales usando los valores de train."""
    df = df.copy()

    if "Delivery_person_Age" in df.columns:
        age_fill = (
            df["Ciudad_codigo"]
            .map(artifacts["age_median_by_city"])
            .fillna(artifacts["age_median_global"])
        )
        df["Delivery_person_Age"] = df["Delivery_person_Age"].fillna(age_fill)

    if "Delivery_person_Ratings" in df.columns:
        ratings_fill = (
            df["Ciudad_codigo"]
            .map(artifacts["ratings_median_by_city"])
            .fillna(artifacts["ratings_median_global"])
        )
        df["Delivery_person_Ratings"] = df["Delivery_person_Ratings"].fillna(
            ratings_fill
        )

    if "multiple_deliveries" in df.columns:
        df["multiple_deliveries"] = df["multiple_deliveries"].fillna(
            artifacts["multiple_deliveries_mode"]
        )

    if "Festival" in df.columns:
        df["Festival"] = df["Festival"].fillna("No")

    if "City" in df.columns:
        df["City"] = df["City"].fillna("Unknown")

    if "Road_traffic_density" in df.columns:
        df["Road_traffic_density"] = df["Road_traffic_density"].fillna(
            artifacts["traffic_mode"]
        )

    if "Weather_conditions" in df.columns:
        df["Weather_conditions"] = df["Weather_conditions"].fillna(
            artifacts["weather_mode"]
        )

    prep_median = artifacts.get("prep_time_median", 10.0)
    mask_orderd_na = df["Time_Orderd_min"].isna()
    if mask_orderd_na.any():
        picked_vals = df.loc[mask_orderd_na, "Time_Order_picked_min"].fillna(730.0)
        df.loc[mask_orderd_na, "Time_Orderd_min"] = (
            picked_vals - prep_median
        ) % 1440

    mask_picked_na = df["Time_Order_picked_min"].isna()
    if mask_picked_na.any():
        df.loc[mask_picked_na, "Time_Order_picked_min"] = (
            df.loc[mask_picked_na, "Time_Orderd_min"] + prep_median
        ) % 1440

    df = df.drop(columns=["Time_Orderd", "Time_Order_picked"], errors="ignore")
    return df



def engineer_features(df, prep_time_median=10.0):
    """
    Pipeline principal de ingeniería de características.
    Recibe el DataFrame limpio y devuelve el DataFrame enriquecido y determinista
    (apto tanto para lotes completos como para inferencia de 1 sola fila).
    """
    df = df.copy()
    df = calculate_distances(df)
    df = calculate_prep_time(df, prep_time_median=prep_time_median)
    df = extract_temporal_features(df)
    df = encode_categoricals(df)
    return df


# Paso B1: Calcular distancias (Haversine y Manhattan corregida)
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en km usando la fórmula de Haversine."""
    earth_radius_km = 6371.0
    lat1_r, lon1_r, lat2_r, lon2_r = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2_r - lat1_r
    dlon = lon2_r - lon1_r
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2.0) ** 2
    )
    a = np.clip(a, 0.0, 1.0)
    c = 2.0 * np.arcsin(np.sqrt(a))
    return earth_radius_km * c


def calculate_distances(df):
    """Calcula dist_haversine_km y dist_manhattan_km ponderada por cos(lat_media)."""
    df = df.copy()
    lat1 = df["Restaurant_latitude"].astype(float)
    lon1 = df["Restaurant_longitude"].astype(float)
    lat2 = df["Delivery_location_latitude"].astype(float)
    lon2 = df["Delivery_location_longitude"].astype(float)

    df["dist_haversine_km"] = haversine_distance(lat1, lon1, lat2, lon2)

    mean_lat_rad = np.radians((lat1 + lat2) / 2.0)
    delta_lat_km = np.abs(lat2 - lat1) * 111.132
    delta_lon_km = np.abs(lon2 - lon1) * 111.320 * np.cos(mean_lat_rad)
    df["dist_manhattan_km"] = delta_lat_km + delta_lon_km

    return df


# Paso B2: Calcular tiempo de preparación
def calculate_prep_time(df, prep_time_median=10.0):
    """Calcula Tiempo_preparacion_min corrigiendo cruces de medianoche y valores >180 min."""
    df = df.copy()
    prep = df["Time_Order_picked_min"] - df["Time_Orderd_min"]
    prep = np.where(prep < 0, prep + 1440.0, prep)
    prep = pd.Series(prep, index=df.index, dtype=float)
    prep = prep.where(prep <= 180.0, np.nan).fillna(float(prep_time_median))
    df["Tiempo_preparacion_min"] = prep
    return df


# Paso B3: Extraer variables temporales y cíclicas
def _assign_franja_horaria(hour):
    if 0 <= hour < 6:
        return "Madrugada"
    if 6 <= hour < 12:
        return "Manana"
    if 12 <= hour < 16:
        return "Almuerzo_Pico"
    if 16 <= hour < 19:
        return "Tarde_Valle"
    return "Cena_Pico"


def extract_temporal_features(df):
    """Extrae día de semana, fin de semana, hora, seno/coseno cíclicos y franja horaria."""
    df = df.copy()
    order_dt = pd.to_datetime(df["Order_Date"], format="%d-%m-%Y", errors="coerce")

    df["dia_semana"] = order_dt.dt.dayofweek.fillna(3).astype(int)
    df["es_fin_semana"] = (df["dia_semana"] >= 5).astype(int)

    df["hora_pedido"] = ((df["Time_Orderd_min"] // 60) % 24).astype(float)
    df["hora_sin"] = np.sin(2.0 * np.pi * df["hora_pedido"] / 24.0)
    df["hora_cos"] = np.cos(2.0 * np.pi * df["hora_pedido"] / 24.0)
    df["Franja_horaria"] = df["hora_pedido"].astype(int).map(_assign_franja_horaria)

    df = df.drop(
        columns=["Order_Date", "Time_Orderd_min", "Time_Order_picked_min"],
        errors="ignore",
    )
    return df


# Paso B4: Codificación ordinal y One-Hot determinista
def encode_categoricals(df):
    """Aplica codificación ordinal lógica y One-Hot determinista sin drop por constancia."""
    df = df.copy()

    traffic_map = {"Low": 1, "Medium": 2, "High": 3, "Jam": 4}
    city_ord_map = {"Semi-Urban": 1, "Urban": 2, "Metropolitan": 3, "Unknown": 2}
    weather_map = {
        "Sunny": 0,
        "Cloudy": 1,
        "Windy": 2,
        "Fog": 3,
        "Sandstorms": 4,
        "Stormy": 5,
    }

    if "Weather_conditions" in df.columns:
        weather_vals = df["Weather_conditions"].astype(str).str.strip()
        for cat in ["Cloudy", "Fog", "Sandstorms", "Stormy", "Windy"]:
            df[f"Weather_{cat}"] = (weather_vals == cat).astype(int)
        df["Weather_conditions"] = weather_vals.map(weather_map).fillna(0).astype(int)

    if "Road_traffic_density" in df.columns:
        traffic_vals = df["Road_traffic_density"].astype(str).str.strip()
        df["Road_traffic_density"] = traffic_vals.map(traffic_map).fillna(2).astype(int)

    if "City" in df.columns:
        city_vals = (
            df["City"].astype(str).str.strip().replace("Metropolitian", "Metropolitan")
        )
        df["City_ord"] = city_vals.map(city_ord_map).fillna(2).astype(int)
        for cat in ["Semi-Urban", "Urban", "Unknown"]:
            df[f"City_{cat}"] = (city_vals == cat).astype(int)

    if "Type_of_order" in df.columns:
        order_vals = df["Type_of_order"].astype(str).str.strip()
        for cat in ["Drinks", "Meal", "Snack"]:
            df[f"Type_of_order_{cat}"] = (order_vals == cat).astype(int)

    if "Type_of_vehicle" in df.columns:
        vehicle_vals = df["Type_of_vehicle"].astype(str).str.strip()
        for cat in ["motorcycle", "scooter", "electric_scooter"]:
            df[f"Type_of_vehicle_{cat}"] = (vehicle_vals == cat).astype(int)

    if "Festival" in df.columns:
        festival_vals = df["Festival"].astype(str).str.strip()
        df["Festival_Yes"] = (festival_vals == "Yes").astype(int)

    if "Franja_horaria" in df.columns:
        franja_vals = df["Franja_horaria"].astype(str).str.strip()
        for cat in ["Manana", "Almuerzo_Pico", "Tarde_Valle", "Cena_Pico"]:
            df[f"Franja_{cat}"] = (franja_vals == cat).astype(int)

    df = df.drop(
        columns=["Type_of_order", "Type_of_vehicle", "Festival", "City", "Franja_horaria"],
        errors="ignore",
    )
    return df


**Paso 2: Cargar los datos crudos (.parquet)**

In [2]:
df_train = pd.read_parquet('../data/01_raw/train-00000-of-00001.parquet')
df_test = pd.read_parquet('../data/01_raw/test-00000-of-00001.parquet')

print(f"Train original: {df_train.shape}")
print(f"Test original:  {df_test.shape}")

Train original: (36467, 20)
Test original:  (9117, 20)


**Paso 3: Aplicación de la Limpieza (`fit` exclusivo en `train`) y Feature Engineering**
- Se limpian espacios en blanco, se normalizan nulos y se extrae `Ciudad_codigo` desde `Delivery_person_ID`.
- Se reparan edades ilegales (`< 18` años -> `NaN`) y se truncan ratings fuera de rango (`> 5.0` -> `5.0`).
- Se reparan horas en formato decimal de Excel y `24:XX:00`.
- Los estadísticos de imputación y centroides de ciudad se ajustan **exclusivamente sobre `train`** y se guardan en `models/imputer_artifacts.joblib`.
- Se corrigen coordenadas negativas con `abs()` y se trasladan los registros `(0,0)` al centroide de su `Ciudad_codigo` preservando $(\Delta \phi, \Delta \lambda)$, sin descartar filas.
- Se calculan distancias Haversine y Manhattan ponderada por $\cos(\bar{\phi})$, tiempo de preparación, variables temporales cíclicas, franja horaria y codificación determinista.

In [3]:
# Limpiar Train ajustando artefactos exclusivamente en Train
df_train_clean, artifacts = clean_data(df_train, fit=True, return_artifacts=True)
df_train_fe = engineer_features(
    df_train_clean, prep_time_median=artifacts['prep_time_median']
)

# Limpiar Test reutilizando exclusivamente los artefactos de Train (sin Data Leakage)
df_test_clean = clean_data(df_test, artifacts=artifacts, fit=False)
df_test_fe = engineer_features(
    df_test_clean, prep_time_median=artifacts['prep_time_median']
)

print(f"Train procesado: {df_train_fe.shape} | Nulos totales: {int(df_train_fe.isna().sum().sum())}")
print(f"Test procesado:  {df_test_fe.shape} | Nulos totales: {int(df_test_fe.isna().sum().sum())}")
print(f"Diferencia de columnas: {set(df_train_fe.columns) ^ set(df_test_fe.columns)}")

Train procesado: (36467, 42) | Nulos totales: 0
Test procesado:  (9117, 42) | Nulos totales: 0
Diferencia de columnas: set()


**Paso 4: Verificar inferencia de 1 sola fila**

In [4]:
# Simular la llegada de un único pedido nuevo
single_raw = df_test.iloc[[0]].copy()
single_clean = clean_data(single_raw, artifacts=artifacts, fit=False)
single_fe = engineer_features(
    single_clean, prep_time_median=artifacts['prep_time_median']
)

assert single_fe.shape == (1, df_train_fe.shape[1]), 'Columnas inconsistentes en inferencia individual'
assert list(single_fe.columns) == list(df_train_fe.columns), 'Orden/nombres de columnas distintos'
assert int(single_fe.isna().sum().sum()) == 0, 'Nulos detectados en inferencia individual'

print(f"Inferencia de 1 fila validada: {single_fe.shape} | Columnas idénticas a train: {list(single_fe.columns) == list(df_train_fe.columns)}")

Inferencia de 1 fila validada: (1, 42) | Columnas idénticas a train: True


**Paso 5: Guardar los datos preparados en `02_interim/` y `03_processed/`**

In [5]:
os.makedirs('../data/02_interim', exist_ok=True)
os.makedirs('../data/03_processed', exist_ok=True)

df_train_fe.to_parquet('../data/02_interim/train_fe.parquet', index=False)
df_test_fe.to_parquet('../data/02_interim/test_fe.parquet', index=False)

# Copia sincronizada en 03_processed para compatibilidad con el flujo previo
df_train_fe.to_parquet('../data/03_processed/train_processed.parquet', index=False)
df_test_fe.to_parquet('../data/03_processed/test_processed.parquet', index=False)

print('¡Datos guardados exitosamente!')
print(f"   - data/02_interim/train_fe.parquet          {df_train_fe.shape}")
print(f"   - data/02_interim/test_fe.parquet           {df_test_fe.shape}")
print(f"   - data/03_processed/train_processed.parquet {df_train_fe.shape}")
print(f"   - data/03_processed/test_processed.parquet  {df_test_fe.shape}")
print(f"   - models/imputer_artifacts.joblib           (ciudades={len(artifacts['city_centroids'])})")

¡Datos guardados exitosamente!
   - data/02_interim/train_fe.parquet          (36467, 42)
   - data/02_interim/test_fe.parquet           (9117, 42)
   - data/03_processed/train_processed.parquet (36467, 42)
   - data/03_processed/test_processed.parquet  (9117, 42)
   - models/imputer_artifacts.joblib           (ciudades=22)


**Paso 6: Auditoría rápida de rangos y vista previa**

In [6]:
summary_cols = [
    'Delivery_person_Age', 'Delivery_person_Ratings',
    'dist_haversine_km', 'dist_manhattan_km',
    'Tiempo_preparacion_min', 'Road_traffic_density', 'City_ord'
]
display(df_train_fe[summary_cols].describe().round(2))
df_train_fe.head()

,Delivery_person_Age,Delivery_person_Ratings,dist_haversine_km,dist_manhattan_km,Tiempo_preparacion_min,Road_traffic_density,City_ord
count,36467.00,36467.00,36467.00,36467.00,36467.00,36467.00,36467.00
mean,29.58,4.64,9.71,13.73,9.99,2.36,2.74
std,5.67,0.32,5.60,7.91,4.00,1.25,0.44
min,20.00,1.00,1.47,2.07,5.00,1.00,1.00
25%,25.00,4.60,4.66,6.59,5.00,1.00,2.00
50%,30.00,4.70,9.19,13.00,10.00,2.00,3.00
75%,34.00,4.80,13.63,19.27,15.00,4.00,3.00
max,50.00,5.00,20.97,29.62,15.00,4.00,3.00


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Weather_conditions,Road_traffic_density,...,Type_of_order_Meal,Type_of_order_Snack,Type_of_vehicle_motorcycle,Type_of_vehicle_scooter,Type_of_vehicle_electric_scooter,Festival_Yes,Franja_Manana,Franja_Almuerzo_Pico,Franja_Tarde_Valle,Franja_Cena_Pico
0,0x6bbb,INDORES02DEL01,23.0,4.5,22.651847,75.881991,22.721847,75.951991,5,1,...,0,0,1,0,0,0,0,0,0,1
1,0x190a,MYSRES20DEL01,39.0,4.3,12.337978,76.616792,12.377978,76.656792,1,2,...,0,0,1,0,0,0,0,1,0,0
2,0x120c,MYSRES07DEL02,37.0,4.8,12.325461,76.632278,12.375461,76.682278,2,1,...,1,0,0,1,0,0,0,0,0,1
3,0x6016,SURRES12DEL03,30.0,4.8,21.183434,72.814492,21.213434,72.844492,4,2,...,0,1,1,0,0,0,0,0,1,0
4,0x9ca0,VADRES13DEL02,29.0,4.6,22.310237,73.158921,22.360237,73.208921,1,4,...,0,0,0,1,0,0,0,0,0,1
